# Módulo 2 · 3/3 — Web Scraping con BeautifulSoup y Scrapy

En esta clase vamos a cubrir:

1. ¿Qué es el web scraping? Y cómo funciona una página HTML
2. `requests` para descargar el contenido
3. `BeautifulSoup` para analizar y navegar el HTML
4. Métodos clave: `find`, `find_all`, `select`, `get_text`, `attrs`
5. `Scrapy`: arquitectura (arañas, items, pipelines) y cuándo usarlo
6. Ética del scraping (robots.txt, términos de servicio)
7. Un quiz interactivo para repasar

> Trabajaremos con un **HTML local de muestra** en `datos/web/productos.html` para no depender de la red.


---
## 1. ¿Qué es el web scraping?

El **web scraping** es la extracción automatizada de datos de sitios web. Una página web está escrita en **HTML** (etiquetas anidadas). Extraer datos de ese HTML a mano es tedioso; por eso usamos bibliotecas como **requests** + **BeautifulSoup**, o frameworks como **Scrapy** para proyectos a escala.

**Flujo típico (bs4):**
1. Descargar el HTML con `requests`.
2. Analizarlo con `BeautifulSoup`.
3. Localizar elementos con `find`/`find_all`/`select`.
4. Extraer texto (`get_text`) o atributos (`attrs`).


---
## 2. Descargar contenido con requests

En clase usamos un HTML **local** (`datos/web/productos.html`) para que el código corra sin internet. En un caso real usaríamos `requests.get(url)` para obtener el contenido de una página.

```python
import requests
respuesta = requests.get("https://ejemplo.com/pagina")
html = respuesta.text
```

> ⚠️ En este notebook leemos el archivo local directamente para que todo funcione offline.


In [1]:
from pathlib import Path
from bs4 import BeautifulSoup

html = Path("datos/web/productos.html").read_text(encoding="utf-8")

soup = BeautifulSoup(html, "html.parser")

print("Título de la página:", soup.title.get_text())
print("Encabezado h1:", soup.find("h1").get_text())
print("Nº de etiquetas a:", len(soup.find_all("a")))

Título de la página: Tienda de Sample
Encabezado h1: Nuestra tienda
Nº de etiquetas a: 1


In [2]:
soup

<!DOCTYPE html>

<html>
<head><title>Tienda de Sample</title></head>
<body>
<h1>Nuestra tienda</h1>
<div class="producto">
<h2 class="nombre">Auriculares Pro</h2>
<span class="precio">$99.99</span>
<p class="descripcion">Excelente sonido envolvente</p>
</div>
<div class="producto">
<h2 class="nombre">Teclado Mecanico</h2>
<span class="precio">$129.50</span>
<p class="descripcion">Switch rojos, retroiluminado</p>
</div>
<div class="producto">
<h2 class="nombre">Mouse Gamer</h2>
<span class="precio">$45.00</span>
<p class="descripcion">Alta precision 16000 DPI</p>
</div>
<a href="https://www.example.com">Visita el sitio</a>
</body>
</html>

---
## 3. Métodos clave de BeautifulSoup

- `soup.find(...)` → el **primer** elemento que coincide.
- `soup.find_all(...)` → **todos** los elementos que coinciden (lista).
- `soup.select(selector_css)` / `soup.select_one(...)` → usa **selectores CSS** (`.clase`, `#id`, `etiqueta`).
- `.get_text()` → texto limpio sin etiquetas.
- `elemento.attrs` → diccionario con los atributos (p. ej. `href`).
- En Python usamos `class_=` (con guión bajo) para evitar la palabra reservada `class`.


In [3]:
from pathlib import Path
from bs4 import BeautifulSoup

html = Path("datos/web/productos.html").read_text(encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")

# find_all: todos los productos (div con clase 'producto')
productos = soup.find_all("div", class_="producto")
print("Productos encontrados:", len(productos))

# Recorrer y extraer nombre, precio y descripción de cada uno
for idx, prod in enumerate(productos, start=1):
    nombre = prod.find("h2", class_="nombre").get_text()
    precio = prod.find("span", class_="precio").get_text()
    desc = prod.find("p", class_="descripcion").get_text()
    print(f"{idx}. {nombre} - {precio} - {desc}")

Productos encontrados: 3
1. Auriculares Pro - $99.99 - Excelente sonido envolvente
2. Teclado Mecanico - $129.50 - Switch rojos, retroiluminado
3. Mouse Gamer - $45.00 - Alta precision 16000 DPI


In [4]:
from pathlib import Path
from bs4 import BeautifulSoup

html = Path("datos/web/productos.html").read_text(encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")

# selectores CSS: .nombre dentro de .producto
nombres = [n.get_text() for n in soup.select(".producto .nombre")]
print("Nombres (CSS selector):", nombres)

# select_one devuelve solo el primero
primero = soup.select_one(".producto .precio")
print("Primer precio:", primero.get_text())

# Extraer atributos: todos los href de los enlaces <a>
enlaces = [a.attrs.get("href") for a in soup.find_all("a")]
print("Enlaces (href):", enlaces)

Nombres (CSS selector): ['Auriculares Pro', 'Teclado Mecanico', 'Mouse Gamer']
Primer precio: $99.99
Enlaces (href): ['https://www.example.com']


---
## 4. Scrapy: framework de scraping a escala

Para proyectos **pequeños** (una página, pocos datos) `requests` + `BeautifulSoup` bastan. Cuando el problema es **grande y complejo**, usamos **Scrapy**, un framework que organiza el scraping en componentes:

- **Arañas (spiders)**: navegan sitios y definen qué datos extraer.
- **Items**: definen la **estructura** de los datos recolectados (campos y tipos).
- **Tuberías (pipelines)**: limpian, validan y **exportan** los datos (CSV, JSON, base de datos).

Ventajas de Scrapy: arquitectura **asíncrona** (múltiples requests concurrentes), manejo de **paginación**, cookies, reintentos y **estrangulamiento** (control del ritmo) para no sobrecargar servidores.


In [5]:
# Concepto de componentes de Scrapy (no ejecuta una araña real aquí).
# En la práctica: se crea un proyecto con 'scrapy startproject mi_proyecto' desde la terminal
# (CLI) y las spiders se guardan en la carpeta spiders/. Aquí vemos un resumen conceptual.

estructura_concepto = {
    "arañas (spiders)": "navegan el sitio y extraen datos",
    "items": "definen los campos (p.ej. nombre, precio, imagen)",
    "tuberías (pipelines)": "limpian, validan y exportan (CSV/JSON/BD)",
}

for componente, funcion in estructura_concepto.items():
    print(f"- {componente}: {funcion}")

print()
print("Flujo típico desde la terminal:")
print("  scrapy startproject mi_tienda_de_datos")
print("  scrapy genspider productos www.ejemplo.com")
print("  scrapy crawl productos -o productos.csv")

- arañas (spiders): navegan el sitio y extraen datos
- items: definen los campos (p.ej. nombre, precio, imagen)
- tuberías (pipelines): limpian, validan y exportan (CSV/JSON/BD)

Flujo típico desde la terminal:
  scrapy startproject mi_tienda_de_datos
  scrapy genspider productos www.ejemplo.com
  scrapy crawl productos -o productos.csv


---
## 5. Paginación y ética

**Paginación**: cuando la información está repartida en varias páginas, se suele identificar el **patrón de las URL** (`?page=1`, `?page=2`...) y generar una solicitud por cada página.

**Ética del scraping**:
- Respetar el archivo **robots.txt** del sitio.
- Revisar los **términos de servicio** (TOS).
- No hacer **peticiones excesivas** que sobrecarguen el servidor (usar estrangulamiento/delays).
- Scrapy integra `ROBOTSTXT_OBEY` y `DOWNLOAD_DELAY` para esto.


In [6]:
# Comprobar si existe un robots.txt local de muestra (concepto de ética)
from pathlib import Path

robots = Path("datos/web/robots.txt")
if robots.exists():
    print(robots.read_text(encoding="utf-8"))
else:
    # Creamos un ejemplo de robots.txt para ilustrar el concepto
    robots.write_text(
        "User-agent: *\nDisallow: /admin/\nDisallow: /privado/\n",
        encoding="utf-8",
    )
    print("robots.txt de muestra creado:")
    print(robots.read_text(encoding="utf-8"))

User-agent: *
Disallow: /admin/
Disallow: /privado/



---
# Quiz interactivo: Web Scraping

Respondé cada pregunta y pulsá **Ver respuesta**. Hay preguntas de una sola opción y de múltiple respuesta.


In [7]:
# --- Configuración del quiz ---
import sys
from pathlib import Path

RAIZ = Path.cwd()
while RAIZ.parent != RAIZ and not (RAIZ / "quiz").exists():
    RAIZ = RAIZ.parent
if str(RAIZ) not in sys.path:
    sys.path.append(str(RAIZ))

from quiz.quiz import aplicar_estilos, crear_quiz, crear_quiz_multiple
aplicar_estilos()

import json
with open(Path.cwd() / "Modulo_2_WebScraping_preguntas.json", encoding="utf-8") as fh:
    respuestas = json.load(fh)

print("Quiz listo:", ", ".join(respuestas.keys()))

Quiz listo: q1, q2, q3, q4, q5, q6, q7, q8, q9, q10


### Pregunta 1

¿Qué biblioteca convierte el HTML de una página en una **estructura de árbol** que facilita extraer datos?

In [8]:
q1_data = respuestas["q1"]
crear_quiz(q1_data["opciones"], q1_data["correcta"], q1_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) requests', 'A'), ('B) beautifulsoup4', 'B'), ('C) scra…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

In [11]:
import requests

In [12]:
emi=requests.get("https://emipaz.com.ar")

In [18]:
sopa_emi = BeautifulSoup(emi.text, "html.parser")

### Pregunta 2

¿Qué método devuelve una lista con **TODOS** los elementos que coinciden con los criterios?

In [17]:
q2_data = respuestas["q2"]
crear_quiz(q2_data["opciones"], q2_data["correcta"], q2_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) find()', 'A'), ('B) find_all()', 'B'), ('C) select_one…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

In [ ]:
#tabla-rowcolspan-int > tbody > tr:nth-child(2) > td:nth-child(1) > a = sopa_emi.find_all("p")
print("Nº de párrafos en emi:", len(parrafos))

Nº de párrafos en emi: 25


In [21]:
parrafos[:5]

[<p>Tu mensaje llegÃ³ bien. Cuando lo lea...<br/><strong>capaz te llamo</strong> ð</p>,
 <p class="modal-subtext">â Emiliano</p>,
 <p><strong>ð¥ï¸ Infraestructura:</strong> Linux (WSL en Windows 10)</p>,
 <p><strong>ð DNS:</strong> Cloudflare</p>,
 <p><strong>ð³ Contenedores:</strong> Docker</p>]

### Pregunta 3

Tenés ya seleccionado un elemento (un párrafo de una reseña). ¿Qué método usás para obtener su **texto limpio** sin etiquetas?

In [22]:
q3_data = respuestas["q3"]
crear_quiz(q3_data["opciones"], q3_data["correcta"], q3_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(("A) find('p')", 'A'), ("B) select_one('.review-text')", 'B…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

In [23]:
parrafos[0].get_text()

'Tu mensaje llegÃ³ bien. Cuando lo lea...capaz te llamo ð\x9f\x98\x84'

### Pregunta 4

El precio de un producto está en `<span class="precio">$99.99</span>`. Dado el objeto `soup`, ¿qué código lo extrae correctamente?

In [24]:
q4_data = respuestas["q4"]
crear_quiz(q4_data["opciones"], q4_data["correcta"], q4_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(("A) soup.find('div', class_='precio').get_text()", 'A'), (…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 5

¿Cuáles de los siguientes son **componentes** del framework Scrapy? (Seleccioná todas las correctas)

In [25]:
q5_data = respuestas["q5"]
crear_quiz_multiple(q5_data["opciones"], q5_data["correctas"], q5_data["explicacion"])

Checkbox(value=False, description='A) Arañas (spiders) que navegan y extraen datos', indent=False, layout=Layo…

Checkbox(value=False, description='B) Items que definen la estructura de los datos', indent=False, layout=Layo…

Checkbox(value=False, description='C) Tuberías (pipelines) que procesan y exportan los datos', indent=False, l…

Checkbox(value=False, description='D) Hojas de estilo CSS del sitio', indent=False, layout=Layout(width='100%'…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 6

En Scrapy, ¿cuál es la función principal de una **araña** (spider)?

In [26]:
q6_data = respuestas["q6"]
crear_quiz(q6_data["opciones"], q6_data["correcta"], q6_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) Evitar el acceso no autorizado a un sitio', 'A'), ('B)…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 7

¿Cuáles de las siguientes son **prácticas** de web scraping **ético**? (Seleccioná todas las correctas)

In [27]:
q7_data = respuestas["q7"]
crear_quiz_multiple(q7_data["opciones"], q7_data["correctas"], q7_data["explicacion"])

Checkbox(value=False, description='A) Respetar el archivo robots.txt del sitio', indent=False, layout=Layout(w…

Checkbox(value=False, description='B) Revisar los Términos de servicio del sitio', indent=False, layout=Layout…

Checkbox(value=False, description='C) Recolectar datos lo más rápido posible para minimizar carga', indent=Fal…

Checkbox(value=False, description='D) Evitar peticiones excesivas que sobrecarguen el servidor', indent=False,…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 8

Para obtener la URL (`href`) de una etiqueta `<a>`, ¿qué atributo/estructura usás?

In [28]:
q8_data = respuestas["q8"]
crear_quiz(q8_data["opciones"], q8_data["correcta"], q8_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) tag.string', 'A'), ("B) tag.attrs['href']", 'B'), ('C)…

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 9

Querés que una araña de Scrapy recorra un sitio con **varias páginas** de contenido. ¿Qué técnica es clave para la **paginación**?

In [29]:
q9_data = respuestas["q9"]
crear_quiz(q9_data["opciones"], q9_data["correcta"], q9_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) Definir todas las URL en start_urls', 'A'), ('B) Usar …

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

### Pregunta 10

Un sitio carga su contenido con **JavaScript/AJAX** después de la carga inicial. ¿Qué limitación tiene `BeautifulSoup` aquí?

In [30]:
q10_data = respuestas["q10"]
crear_quiz(q10_data["opciones"], q10_data["correcta"], q10_data["explicacion"])

RadioButtons(layout=Layout(width='100%'), options=(('A) BeautifulSoup por sí solo no ve contenido cargado con …

Button(description='Ver respuesta', layout=Layout(width='auto'), style=ButtonStyle())

Output()

## ¡Fin del quiz!

Revisá las explicaciones y repasá la arquitectura de Scrapy y los métodos de bs4. Reiniciá y reejecutá el **setup** y las preguntas para reintentar.